In [ ]:
# ============================================================================
# 🧠 脑MRI体素分类 - TabNet迁移学习实现
# 基于PyTorch Lightning的逐步解冻策略
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset
import h5py
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import json
import os
from tqdm import tqdm
import time

# ============================================================================
# Section 1: 数据加载（复用baseline代码）
# ============================================================================

def load_brain_voxel_data():
    """加载并预处理数据"""
    print("📂 加载数据...")
    f = h5py.File('/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat','r')
    arrays = {}
    for k, v in f.items():
        arrays[k] = np.array(v)
    f.close()
    
    train_data = arrays['data'].transpose()
    train_region = arrays['region'].transpose()
    prob_idx = arrays['prob_idx'].transpose()
    
    # 数据分割
    test_indices = np.where(prob_idx == 38)[0]
    train_val_indices = np.where(prob_idx != 38)[0]
    
    test_data = train_data[test_indices, :]
    test_labels = train_region[test_indices, :]
    train_val_data = train_data[train_val_indices, :]
    train_val_labels = train_region[train_val_indices, :]
    
    # 进一步分割
    X_train, X_val, y_train, y_val = train_test_split(
        train_val_data, train_val_labels, 
        test_size=0.2, random_state=42, 
        stratify=np.argmax(train_val_labels, axis=1)
    )
    
    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(test_data)
    
    print(f"训练集: {X_train_scaled.shape}")
    print(f"验证集: {X_val_scaled.shape}")
    print(f"测试集: {X_test_scaled.shape}")
    
    return (X_train_scaled, y_train, 
            X_val_scaled, y_val, 
            X_test_scaled, test_labels, 
            scaler)

# ============================================================================
# Section 2: 自定义TabNet模块（支持逐步解冻）
# ============================================================================

class TabNetEncoder(nn.Module):
    """可控制的TabNet编码器"""
    def __init__(self, input_dim=341, output_dim=64, n_steps=6, 
                 feature_dim=64, virtual_batch_size=128):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.n_steps = n_steps
        
        # 初始批标准化
        self.initial_bn = nn.BatchNorm1d(input_dim)
        
        # 决策步骤（每个步骤可独立冻结）
        self.decision_steps = nn.ModuleList()
        for step in range(n_steps):
            self.decision_steps.append(
                DecisionStep(
                    input_dim if step == 0 else feature_dim,
                    feature_dim,
                    feature_dim,
                    step_num=step
                )
            )
        
        # 最终投影
        self.final_projection = nn.Linear(feature_dim, output_dim)
        
    def forward(self, x, return_attention=False):
        x = self.initial_bn(x)
        
        # 逐步处理
        decision_outputs = []
        attention_masks = []
        
        for i, step in enumerate(self.decision_steps):
            x, mask = step(x)
            decision_outputs.append(x)
            attention_masks.append(mask)
        
        # 聚合输出
        output = torch.mean(torch.stack(decision_outputs), dim=0)
        output = self.final_projection(output)
        
        if return_attention:
            return output, attention_masks
        return output
    
    def freeze_steps(self, steps_to_freeze):
        """冻结指定的决策步骤"""
        for i, step in enumerate(self.decision_steps):
            if i in steps_to_freeze:
                for param in step.parameters():
                    param.requires_grad = False
            else:
                for param in step.parameters():
                    param.requires_grad = True

class DecisionStep(nn.Module):
    """单个决策步骤"""
    def __init__(self, input_dim, output_dim, feature_dim, step_num):
        super().__init__()
        self.step_num = step_num
        
        # 特征变换
        self.feature_transformer = nn.Sequential(
            nn.Linear(input_dim, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU()
        )
        
        # 注意力机制
        self.attention = nn.Sequential(
            nn.Linear(input_dim, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.Tanh()
        )
        
    def forward(self, x):
        # 特征变换
        features = self.feature_transformer(x)
        
        # 注意力掩码
        mask = self.attention(x)
        
        # 应用注意力
        output = features * mask
        
        return output, mask

# ============================================================================
# Section 3: Lightning模块实现
# ============================================================================

class TabNetTransferLearning(pl.LightningModule):
    """TabNet迁移学习Lightning模块"""
    
    def __init__(self, pretrained_path=None, num_classes=102, 
                 learning_rate=1e-3, freeze_stage=0):
        super().__init__()
        self.save_hyperparameters()
        
        # TabNet编码器
        self.encoder = TabNetEncoder(
            input_dim=341,
            output_dim=128,
            n_steps=6,
            feature_dim=64
        )
        
        # 加载预训练权重
        if pretrained_path and os.path.exists(pretrained_path):
            self.load_pretrained_weights(pretrained_path)
        
        # 过渡层
        self.transition = nn.Sequential(
            nn.Linear(128, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 输出头
        self.head = nn.Linear(256, num_classes)
        
        # 设置冻结阶段
        self.freeze_stage = freeze_stage
        self.configure_freezing()
        
        # 训练历史
        self.training_history = {
            'train_loss': [], 'train_f1': [],
            'val_loss': [], 'val_f1': [],
            'test_loss': [], 'test_f1': []
        }
        
    def load_pretrained_weights(self, path):
        """加载预训练权重"""
        try:
            checkpoint = torch.load(path, map_location='cpu')
            
            # 适配权重到我们的编码器
            # 这里需要根据实际的预训练模型结构进行调整
            state_dict = checkpoint if isinstance(checkpoint, dict) else checkpoint.state_dict()
            
            # 部分加载（只加载匹配的层）
            model_dict = self.encoder.state_dict()
            pretrained_dict = {k: v for k, v in state_dict.items() 
                             if k in model_dict and v.shape == model_dict[k].shape}
            
            model_dict.update(pretrained_dict)
            self.encoder.load_state_dict(model_dict)
            
            print(f"✅ 成功加载预训练权重，匹配层数: {len(pretrained_dict)}/{len(model_dict)}")
        except Exception as e:
            print(f"⚠️ 加载预训练权重失败: {e}")
            print("将使用随机初始化")
    
    def configure_freezing(self):
        """配置冻结策略"""
        if self.freeze_stage == 0:  # 全部冻结
            self.encoder.freeze_steps(list(range(6)))
            print("❄️ Stage 0: 冻结所有编码器层")
            
        elif self.freeze_stage == 1:  # 解冻最后2步
            self.encoder.freeze_steps(list(range(4)))
            print("🔓 Stage 1: 解冻最后2个决策步骤")
            
        elif self.freeze_stage == 2:  # 解冻最后4步
            self.encoder.freeze_steps(list(range(2)))
            print("🔓 Stage 2: 解冻最后4个决策步骤")
            
        else:  # 全部解冻
            self.encoder.freeze_steps([])
            print("🔥 Stage 3: 全部解冻")
    
    def forward(self, x):
        # 编码
        encoded = self.encoder(x)
        
        # 过渡
        features = self.transition(encoded)
        
        # 输出
        logits = self.head(features)
        
        return logits
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        # 处理标签
        if y.dim() > 1 and y.size(1) > 1:
            y_indices = torch.argmax(y, dim=1)
        else:
            y_indices = y.long()
        
        # 前向传播
        logits = self(x)
        loss = F.cross_entropy(logits, y_indices)
        
        # 计算F1
        preds = torch.argmax(logits, dim=1)
        f1 = f1_score(y_indices.cpu(), preds.cpu(), average='macro', zero_division=0)
        
        # 记录
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_f1', f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        
        # 处理标签
        if y.dim() > 1 and y.size(1) > 1:
            y_indices = torch.argmax(y, dim=1)
        else:
            y_indices = y.long()
        
        # 前向传播
        logits = self(x)
        loss = F.cross_entropy(logits, y_indices)
        
        # 计算F1
        preds = torch.argmax(logits, dim=1)
        f1 = f1_score(y_indices.cpu(), preds.cpu(), average='macro', zero_division=0)
        
        # 记录
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_f1', f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return {'val_loss': loss, 'val_f1': f1}
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        
        # 处理标签
        if y.dim() > 1 and y.size(1) > 1:
            y_indices = torch.argmax(y, dim=1)
        else:
            y_indices = y.long()
        
        # 前向传播
        logits = self(x)
        loss = F.cross_entropy(logits, y_indices)
        
        # 计算F1
        preds = torch.argmax(logits, dim=1)
        f1 = f1_score(y_indices.cpu(), preds.cpu(), average='macro', zero_division=0)
        
        # 记录
        self.log('test_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('test_f1', f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return {'test_loss': loss, 'test_f1': f1}
    
    def configure_optimizers(self):
        """配置优化器 - 不同层使用不同学习率"""
        # 参数分组
        encoder_params = self.encoder.parameters()
        transition_params = self.transition.parameters()
        head_params = self.head.parameters()
        
        # 根据阶段设置学习率
        if self.freeze_stage == 0:
            # 只训练新层
            params = [
                {'params': transition_params, 'lr': self.hparams.learning_rate},
                {'params': head_params, 'lr': self.hparams.learning_rate}
            ]
        else:
            # 编码器使用更小的学习率
            lr_scale = 0.1 ** self.freeze_stage  # 随着解冻程度降低学习率
            params = [
                {'params': encoder_params, 'lr': self.hparams.learning_rate * lr_scale},
                {'params': transition_params, 'lr': self.hparams.learning_rate},
                {'params': head_params, 'lr': self.hparams.learning_rate}
            ]
        
        optimizer = torch.optim.AdamW(params, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        
        return [optimizer], [scheduler]
    
    def on_train_epoch_end(self):
        """记录训练历史"""
        metrics = self.trainer.logged_metrics
        self.training_history['train_loss'].append(metrics.get('train_loss', 0).item())
        self.training_history['train_f1'].append(metrics.get('train_f1', 0).item())
    
    def on_validation_epoch_end(self):
        """记录验证历史"""
        metrics = self.trainer.logged_metrics
        self.training_history['val_loss'].append(metrics.get('val_loss', 0).item())
        self.training_history['val_f1'].append(metrics.get('val_f1', 0).item())

# ============================================================================
# Section 4: 渐进解冻回调
# ============================================================================

class ProgressiveUnfreezing(pl.Callback):
    """渐进解冻回调"""
    
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.wait_count = 0
        self.best_val_f1 = 0
        self.current_stage = 0
        self.stage_start_epoch = 0
        
    def on_validation_epoch_end(self, trainer, pl_module):
        current_val_f1 = trainer.callback_metrics.get('val_f1', 0).item()
        
        # 检查是否有改善
        if current_val_f1 > self.best_val_f1 + self.min_delta:
            self.best_val_f1 = current_val_f1
            self.wait_count = 0
        else:
            self.wait_count += 1
        
        # 判断是否需要进入下一阶段
        epoch = trainer.current_epoch
        epochs_in_stage = epoch - self.stage_start_epoch
        
        # 条件：1) 达到patience 或 2) 在当前阶段训练足够长
        if (self.wait_count >= self.patience or epochs_in_stage >= 5) and self.current_stage < 3:
            self.current_stage += 1
            self.stage_start_epoch = epoch + 1
            self.wait_count = 0
            
            print(f"\n🚀 进入训练阶段 {self.current_stage}")
            
            # 更新模型的冻结策略
            pl_module.freeze_stage = self.current_stage
            pl_module.configure_freezing()
            
            # 重新配置优化器
            trainer.optimizers = []
            trainer.lr_schedulers = []
            trainer.optimizer_frequencies = []
            trainer.strategy.setup_optimizers(trainer)

# ============================================================================
# Section 5: 主训练流程
# ============================================================================

def train_tabnet_transfer_learning():
    """主训练函数"""
    
    # 1. 加载数据
    print("="*60)
    print("🧠 TabNet迁移学习训练")
    print("="*60)
    
    X_train, y_train, X_val, y_val, X_test, y_test, scaler = load_brain_voxel_data()
    
    # 2. 创建数据加载器
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train), 
        torch.FloatTensor(y_train)
    )
    val_dataset = TensorDataset(
        torch.FloatTensor(X_val), 
        torch.FloatTensor(y_val)
    )
    test_dataset = TensorDataset(
        torch.FloatTensor(X_test), 
        torch.FloatTensor(y_test)
    )
    
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=4)
    
    # 3. 创建模型
    model = TabNetTransferLearning(
        pretrained_path='tabnet_model_test0/network.pt',
        num_classes=102,
        learning_rate=1e-3,
        freeze_stage=0
    )
    
    # 4. 设置回调
    callbacks = [
        ProgressiveUnfreezing(patience=3),
        pl.callbacks.ModelCheckpoint(
            monitor='val_f1',
            mode='max',
            save_top_k=3,
            filename='tabnet-{epoch:02d}-{val_f1:.4f}'
        ),
        pl.callbacks.EarlyStopping(
            monitor='val_f1',
            patience=10,
            mode='max'
        ),
        pl.callbacks.LearningRateMonitor()
    ]
    
    # 5. 创建训练器
    trainer = pl.Trainer(
        max_epochs=40,
        callbacks=callbacks,
        gpus=1 if torch.cuda.is_available() else 0,
        precision=16,
        gradient_clip_val=1.0,
        log_every_n_steps=50,
        val_check_interval=1.0
    )
    
    # 6. 训练
    print("\n🚀 开始训练...")
    start_time = time.time()
    
    trainer.fit(model, train_loader, val_loader)
    
    # 7. 测试
    print("\n🔍 测试最佳模型...")
    trainer.test(model, test_loader)
    
    # 8. 保存结果
    total_time = time.time() - start_time
    print(f"\n✅ 训练完成! 总用时: {total_time/60:.1f}分钟")
    
    # 保存训练历史
    results = {
        'training_history': model.training_history,
        'final_metrics': {
            'test_loss': trainer.logged_metrics.get('test_loss', 0).item(),
            'test_f1': trainer.logged_metrics.get('test_f1', 0).item()
        },
        'total_time': total_time,
        'num_stages': model.freeze_stage + 1
    }
    
    with open('tabnet_transfer_results.json', 'w') as f:
        json.dump(results, f, indent=4)
    
    print(f"📊 最终测试结果:")
    print(f"   Loss: {results['final_metrics']['test_loss']:.4f}")
    print(f"   F1-Macro: {results['final_metrics']['test_f1']:.4f}")
    
    return model, results

# ============================================================================
# Section 6: 结果分析
# ============================================================================

def analyze_results(model, results):
    """分析训练结果"""
    import matplotlib.pyplot as plt
    
    history = results['training_history']
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 训练损失
    axes[0, 0].plot(history['train_loss'], label='Train')
    axes[0, 0].plot(history['val_loss'], label='Val')
    axes[0, 0].set_title('Loss曲线')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # F1分数
    axes[0, 1].plot(history['train_f1'], label='Train')
    axes[0, 1].plot(history['val_f1'], label='Val')
    axes[0, 1].set_title('F1-Macro曲线')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # 阶段分析
    stages = ['Stage 0\n(全冻结)', 'Stage 1\n(解冻2层)', 
              'Stage 2\n(解冻4层)', 'Stage 3\n(全解冻)']
    stage_f1s = []
    
    # 这里需要根据实际训练记录分析各阶段性能
    # 简化示例
    if len(history['val_f1']) >= 4:
        stage_f1s = [
            max(history['val_f1'][:5]) if len(history['val_f1']) > 5 else 0,
            max(history['val_f1'][5:15]) if len(history['val_f1']) > 15 else 0,
            max(history['val_f1'][15:25]) if len(history['val_f1']) > 25 else 0,
            max(history['val_f1'][25:]) if len(history['val_f1']) > 25 else 0
        ]
    
    axes[1, 0].bar(stages[:len(stage_f1s)], stage_f1s)
    axes[1, 0].set_title('各阶段最佳F1')
    axes[1, 0].set_ylabel('Best F1 Score')
    
    # 总结统计
    axes[1, 1].axis('off')
    summary_text = f"""
    训练总结:
    
    最终测试Loss: {results['final_metrics']['test_loss']:.4f}
    最终测试F1: {results['final_metrics']['test_f1']:.4f}
    
    最佳验证F1: {max(history['val_f1']):.4f}
    训练时间: {results['total_time']/60:.1f}分钟
    训练阶段数: {results['num_stages']}
    """
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=12, verticalalignment='center')
    
    plt.tight_layout()
    plt.savefig('tabnet_transfer_results.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================================================
# 执行训练
# ============================================================================

if __name__ == "__main__":
    # 训练模型
    model, results = train_tabnet_transfer_learning()
    
    # 分析结果
    analyze_results(model, results)